With a DB that has an existing `search_results` table (like a combined DB), perform the usual data acquisition steps.

In [2]:
# [*] Setup
from sqlalchemy import create_engine, inspect

# Create the engine
engine = create_engine(f'sqlite:///../data/data.db', echo=True)
inspector = inspect(engine)

# Get API keys
api_key_loc = '../data/keys'
with open(api_key_loc, 'r') as f:
    API_KEYS = f.read().splitlines()
    
import pandas as pd

In [ ]:
# [*] Parallel get channel statistics

from googleapiclient.errors import HttpError
from googleapiclient.discovery import build
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import pandas as pd

import json

tqdm.pandas()

import pandas as pd

# Function to get channel statistics
def get_channel_statistics(channel_id, api_keys, key_index, exhausted_keys):
    while key_index < len(api_keys):
        api_key = api_keys[key_index]
        
        # Skip this key if it's exhausted
        if api_key in exhausted_keys:
            key_index += 1
            continue
        
        youtube = build('youtube', 'v3', developerKey=api_key)
        request = youtube.channels().list(
            part='contentDetails,id,snippet,statistics,topicDetails',
            id=channel_id
        )
        
        try:
            response = request.execute()
            items_data = (
                pd.json_normalize(response.get('items'))
                if 'items' in response and response['items'] 
                else None
            )
            return items_data, key_index  # Return data and the current key index
            
        except HttpError as e:
            if e.resp.status == 403:  # Quota exceeded
                print(f"API key {api_key} has exceeded its quota.")
                exhausted_keys.add(api_key)  # Mark this key as exhausted
                key_index += 1  # Move to the next key
            elif e.resp.status in [400, 404, 503]:
                print(f"Error fetching statistics for {channel_id}: {e}")
                return None, key_index  # Return None for errors we don't want to retry
            else:
                raise e
    print("All API keys are exhausted.")
    return None, key_index  # Return None if no keys are left

# Function to parallelize fetching statistics
def get_channel_statistics_parallel(channel_id_string, api_keys, key_index, exhausted_keys):
    return get_channel_statistics(channel_id_string, api_keys, key_index, exhausted_keys)

from concurrent.futures import ThreadPoolExecutor, as_completed

# Parallelized fetch
def parallel_channel_statistics_fetch(channel_id_strings, api_keys):
    channel_statistics = []
    key_index = 0  # Start with the first API key
    exhausted_keys = set()  # Track exhausted keys
    
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = [
            executor.submit(get_channel_statistics_parallel, channel_id_string, api_keys, key_index, exhausted_keys)
            for channel_id_string in tqdm(channel_id_strings, desc="Fetching Channel Statistics")
        ]

        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing Results"):
            try:
                data, key_index = future.result()  # Get both data and the updated key index
                if data is not None:
                    channel_statistics.append(data)
            except Exception as e:
                print(f"Error processing a channel: {e}")
    
    return pd.concat(channel_statistics, ignore_index=True) if channel_statistics else pd.DataFrame()


# Create the engine
engine = create_engine(f'sqlite:///../data/data.db', echo=True)
inspector = inspect(engine)

if (has_search_results := inspector.has_table('search_results')):
    search_results_table = pd.read_sql_table('search_results', engine, index_col='id')

    # Get unique channel ids and split into batches of 50
    channel_ids = search_results_table['snippet.channelId'].unique()
    
    print(f"Total unique channel IDs: {len(channel_ids)}")
    
    batch_size = 50
    channel_ids = [channel_ids[i:i + batch_size] for i in range(0, len(channel_ids), batch_size)]
    channel_id_strings = [','.join(channel_id_batch) for channel_id_batch in channel_ids]

    # Parallelize the fetching of channel statistics
    channels_df = parallel_channel_statistics_fetch(channel_id_strings, API_KEYS)

    for column in channels_df.columns:
        if channels_df[column].apply(lambda x: isinstance(x, (list, dict))).any():
            channels_df[column] = channels_df[column].apply(json.dumps)
    
    # Write to the SQL table
    channels_df.to_sql('channel_statistics', engine, if_exists='replace', index=False)
    
    # Read back the data for validation
    channels_df = pd.read_sql_table('channel_statistics', engine)
    
    print(channels_df.shape)

In [ ]:
# [*] parallel get playlists

from sqlalchemy import create_engine, inspect
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import pandas as pd
from googleapiclient.errors import HttpError
from googleapiclient.discovery import build

tqdm.pandas()

def get_channel_playlist_df(playlist_id, api_keys, current_key_index, pbar):
    
    next_page_token = None
    channel_playlist_dfs = []
    
    while True:
        try:
            youtube = build('youtube', 'v3', developerKey=api_keys[current_key_index])

            playlist_items_request = youtube.playlistItems().list(
                part='contentDetails,snippet',
                playlistId=playlist_id,
                maxResults=50,
                pageToken=next_page_token
            )
            
            playlist_items_response = playlist_items_request.execute()
            items_df = pd.json_normalize(playlist_items_response['items'])
            channel_playlist_dfs.append(items_df)
            
            try:
                pbar.update(len(items_df))
            except TypeError:
                pass
            
            next_page_token = playlist_items_response.get('nextPageToken')
            
            if not next_page_token:
                return pd.concat(channel_playlist_dfs, ignore_index=True)
            
        except HttpError as e:
            if e.resp.status == 403:  # Quota exceeded
                current_key_index = (current_key_index + 1) % len(api_keys)
                if current_key_index == 0:
                    print("All API keys exhausted.")
                    break
            elif e.resp.status == 404:
                print("Playlist not found.")
                break
            else:
                raise  # Re-raise the exception if it's not a quota error


# Create the engine
engine = create_engine(f'sqlite:///../data/data.db', echo=True)
inspector = inspect(engine)

# Main logic
if (has_channel_statistics := inspector.has_table('channel_statistics')):

    iteration = 0

    while True:
            
        channel_statistics_df = pd.read_sql_table('channel_statistics', engine, index_col='id')
        
        if inspector.has_table('channel_playlists'):
            channel_playlists_df = pd.read_sql_table('channel_playlists', engine)
            channel_playlists_ids = channel_playlists_df['snippet.channelId'].unique()
            channel_statistics_df = channel_statistics_df[~channel_statistics_df.index.isin(channel_playlists_ids)]
            
        len_videos = channel_statistics_df['statistics.videoCount'].astype(int).sum()

        if len_videos == 0:
            print("All videos fetched.")
            break
        else:
            print(f"Iteration {iteration } with {len_videos} videos remaining")

        pbar = tqdm(total=channel_statistics_df['statistics.videoCount'].astype(int).sum(), 
                    desc="Fetching Playlist Items")

        channel_playlists_data = []

        def process_channel(channel_id):
            playlist_id = 'UU' + channel_id[2:]
            print(f"Fetching videos for channel {channel_id} with playlist ID {playlist_id}")
            channel_playlist_df = get_channel_playlist_df(playlist_id, API_KEYS, 0, pbar)
            return channel_playlist_df
        
        with ThreadPoolExecutor(max_workers=5) as executor:  # Adjust the number of workers based on your needs
            futures = {executor.submit(process_channel, channel_id): channel_id for channel_id in channel_statistics_df.index}

            for future in as_completed(futures):
                channel_id = futures[future]
                try:
                    channel_playlist_df = future.result()
                    if channel_playlist_df is not None:
                        channel_playlist_df.to_sql('channel_playlists', engine, if_exists='append', index=False)
                except Exception as e:
                    print(f"Error processing channel {channel_id}: {e}")

        channel_statistics_df = pd.read_sql_table('channel_statistics', engine, index_col='id')
        
        len_videos = channel_statistics_df['statistics.videoCount'].astype(int).sum()
        
        iteration += 1

In [ ]:
%%time

from sqlalchemy import create_engine, inspect
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import pandas as pd
from googleapiclient.errors import HttpError
from googleapiclient.discovery import build

keys = API_KEYS.copy()

def get_video_data_df_by_id_string(video_ids_str, pbar):
    
  try:
    
    youtube = build('youtube', 'v3', developerKey=keys[0])

    request = youtube.videos().list(
        part='contentDetails,id,liveStreamingDetails,snippet,statistics,topicDetails',
        id=video_ids_str
    )
    response = request.execute()
    
    items = response.get('items')
    
    pbar.update(len(items))
    
    items_df = pd.json_normalize(items)
    
    return items_df

  except Exception as e:
    
    keys.pop(0)


from sqlalchemy import text, quoted_name
import json

# Helper function to add missing columns to the database
def add_missing_columns(engine, df, table_name):
    # Get the existing columns in the SQL table using PRAGMA for SQLite
    with engine.connect() as conn:
        existing_columns = conn.execute(text(f"PRAGMA table_info({table_name})")).fetchall()
        existing_columns = [col[1] for col in existing_columns]  # Extract column names

    # Compare with DataFrame columns and add missing ones
    for col in df.columns:
        if col not in existing_columns:
            # Escape the column name if it contains a period
            safe_col = f'"{col}"'  # Use double quotes around the column name to escape periods

            # Determine the column type based on the DataFrame
            col_type = 'TEXT'  # Default to TEXT for simplicity
            if df[col].dtype == 'int64':
                col_type = 'INTEGER'
            elif df[col].dtype == 'float64':
                col_type = 'REAL'

            # Add the new column to the SQL table
            with engine.connect() as conn:
                conn.execute(text(f'ALTER TABLE {table_name} ADD COLUMN {safe_col} {col_type}'))


from sqlalchemy import create_engine, inspect, MetaData, Table, Column

# Create the engine
db_loc = 'sqlite:///../data/data.db'
engine = create_engine(db_loc)
inspector = inspect(engine)

import pandas as pd

from tqdm.notebook import tqdm

import json

# Main logic
if (has_channel_playlists := inspector.has_table('channel_playlists')):
  
  iteration = 0
  
  while True:
    
    chunk_size = 100000

    table_size = pd.read_sql_query("SELECT COUNT(*) FROM channel_playlists", engine).values[0][0]

    chunks = pd.read_sql_table('channel_playlists', engine, chunksize=chunk_size)

    chunked_dfs = [chunk for chunk in tqdm(chunks, total=table_size // chunk_size)]

    channel_playlists_df = pd.concat(chunked_dfs)
    
    if inspector.has_table('video_statistics'):
      video_statistics_df = pd.read_sql_table('video_statistics', engine)
      video_statistics_video_ids = video_statistics_df['id'].unique()
      channel_playlists_df = channel_playlists_df[~channel_playlists_df['contentDetails.videoId'].isin(video_statistics_video_ids)]
        
    len_videos = channel_playlists_df.shape[0]

    if len_videos == 0:
        print("All videos fetched.")
        break
    else:
        print(f"Iteration {iteration } with {len_videos} videos remaining")
        
    pbar = tqdm(total=len_videos, desc="Fetching Video Statistics")
    
    videos_data = []

    video_ids = channel_playlists_df['snippet.resourceId.videoId'].unique()
    
    # break into batches of 50
    batch_size = 50
    video_ids = [','.join(video_ids[i:i + batch_size]) for i in range(0, len(video_ids), batch_size)]

    with ThreadPoolExecutor(max_workers=8) as executor:
      futures = {executor.submit(get_video_data_df_by_id_string, video_ids_str, pbar): video_ids_str for video_ids_str in tqdm(video_ids)}
      for future in tqdm(as_completed(futures), total=len(futures), desc="Processing Results"):
        video_ids_str = futures[future]
        video_data_df = future.result()
        if video_data_df is not None:
          for col in video_data_df.columns:
            if video_data_df[col].apply(lambda x: isinstance(x, (list, dict))).any():
              video_data_df[col] = video_data_df[col].apply(json.dumps)
          add_missing_columns(engine, video_data_df, 'video_statistics')
          video_data_df.to_sql('video_statistics', engine, if_exists='append', index=False)
          
vdf = pd.read_sql_table('video_statistics', engine)

print(vdf.head())

print(vdf.shape)

In [7]:
# drop video_statistics table

import sqlite3

# Create a connection
conn = sqlite3.connect('../data/db/fn_m3_split.db')

# Create a cursor object using the cursor() method
cursor = conn.cursor()

# Drop the table if it exists
cursor.execute("DROP TABLE IF EXISTS video_statistics")

# Commit the change
conn.commit()

# Close the connection
conn.close()

In [ ]:
vdf = pd.concat(vdfs, ignore_index=True).set_index('id')
vdf

In [ ]:
from sqlalchemy import create_engine, inspect, MetaData, Table, Column
from sqlalchemy.types import String
from tqdm.notebook import tqdm
import pandas as pd
from googleapiclient.errors import HttpError
from googleapiclient.discovery import build
import json

def get_video_data_df_by_id_string(video_ids_str, api_keys, current_key_index, pbar):
    
  try:
    youtube = build('youtube', 'v3', developerKey=api_keys[current_key_index])

    request = youtube.videos().list(
        part='contentDetails,id,liveStreamingDetails,snippet,statistics,topicDetails',
        id=video_ids_str
    )
    response = request.execute()
    
    items = response.get('items')
    
    pbar.update(len(items))
    
    items_df = pd.json_normalize(items)
    
    return items_df
    
  except HttpError as e:
    if e.resp.status == 403:  # Quota exceeded
      current_key_index = (current_key_index + 1) % len(api_keys)
      if current_key_index == 0:
        print("All API keys exhausted.")
    elif e.resp.status == 404:
      print("Playlist not found.")
    else:
      raise  # Re-raise the exception if it's not a quota error

def add_missing_columns(engine, table_name, df):
    """Add missing columns to the table if they don't exist."""
    # Reflect the table to check its existing columns
    inspector = inspect(engine)
    existing_columns = [col['name'] for col in inspector.get_columns(table_name)]
    
    # Find columns in DataFrame that are not in the table
    missing_columns = set(df.columns) - set(existing_columns)
    
    # Add missing columns dynamically
    if missing_columns:
        with engine.connect() as conn:
            for column in missing_columns:
                # Add the missing column as TEXT (String type) by default
                alter_stmt = f'ALTER TABLE {table_name} ADD COLUMN "{column}" TEXT'
                conn.execute(alter_stmt)
                print(f"Added missing column: {column}")

# Create the engine
inspector = inspect(engine)

# Main logic
if (has_channel_playlists := inspector.has_table('channel_playlists')):
  
  iteration = 0
  
  while True:
    
    channel_playlists_df = pd.read_sql_table('channel_playlists', engine)
  
    if inspector.has_table('video_statistics'):
      video_statistics_df = pd.read_sql_table('video_statistics', engine)
      video_statistics_video_ids = video_statistics_df['id'].unique()
      channel_playlists_df = channel_playlists_df[~channel_playlists_df['contentDetails.videoId'].isin(video_statistics_video_ids)]

    len_videos = channel_playlists_df.shape[0]

    if len_videos == 0:
        print("All videos fetched.")
        break
    else:
        print(f"Iteration {iteration} with {len_videos} videos remaining")
        
    pbar = tqdm(total=len_videos, desc="Fetching Video Statistics")
    
    videos_data = []

    video_ids = channel_playlists_df['snippet.resourceId.videoId'].unique()
    
    # break into batches of 50
    batch_size = 50
    video_ids = [','.join(video_ids[i:i + batch_size]) for i in range(0, len(video_ids), batch_size)]

    # Sequential execution for each batch of video IDs
    for video_ids_str in video_ids:
      try:
        video_data_df = get_video_data_df_by_id_string(video_ids_str, API_KEYS, 0, pbar)
        if video_data_df is not None:
          
          # Ensure JSON fields are properly formatted
          for column in video_data_df.columns:
            if video_data_df[column].apply(lambda x: isinstance(x, (list, dict))).any():
              video_data_df[column] = video_data_df[column].apply(json.dumps)
          
          # Add missing columns before inserting the data
          add_missing_columns(engine, 'video_statistics', video_data_df)
          
          # Save the data to the database
          video_data_df.to_sql('video_statistics', engine, if_exists='append', index=False)
      except Exception as e:
        print(f"Error processing videos {video_ids_str}: {e}")
    
    iteration += 1
    pbar.close()

In [ ]:
from sqlalchemy import create_engine, inspect, MetaData, Table, Column
from sqlalchemy.types import String
import pandas as pd
from googleapiclient.errors import HttpError
from googleapiclient.discovery import build
import json
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm

def get_video_data_df_by_id_string(video_ids_str, api_keys):
    current_key_index = 0
    while True:
        try:
            youtube = build('youtube', 'v3', developerKey=api_keys[current_key_index])
            request = youtube.videos().list(
                part='contentDetails,id,liveStreamingDetails,snippet,statistics,topicDetails',
                id=video_ids_str
            )
            response = request.execute()
            items = response.get('items')
            items_df = pd.json_normalize(items)
            return items_df
        except HttpError as e:
            if e.resp.status == 403:  # Quota exceeded
                current_key_index = (current_key_index + 1) % len(api_keys)
                if current_key_index == 0:
                    print("All API keys exhausted.")
                    raise Exception("All API keys exhausted.")
            elif e.resp.status == 404:
                print("Playlist not found.")
                return None
            else:
                raise  # Re-raise the exception if it's not a quota error

def add_missing_columns(engine, table_name, df):
    """Add missing columns to the table if they don't exist."""
    inspector = inspect(engine)
    existing_columns = [col['name'] for col in inspector.get_columns(table_name)]
    missing_columns = set(df.columns) - set(existing_columns)
    if missing_columns:
        with engine.connect() as conn:
            for column in missing_columns:
                alter_stmt = f'ALTER TABLE {table_name} ADD COLUMN "{column}" TEXT'
                conn.execute(alter_stmt)
                print(f"Added missing column: {column}")

def process_batch(video_ids_str_api_keys):
    video_ids_str, api_keys = video_ids_str_api_keys
    try:
        video_data_df = get_video_data_df_by_id_string(video_ids_str, api_keys)
        if video_data_df is not None:
            # Ensure JSON fields are properly formatted
            for column in video_data_df.columns:
                if video_data_df[column].apply(lambda x: isinstance(x, (list, dict))).any():
                    video_data_df[column] = video_data_df[column].apply(json.dumps)
            return video_data_df
    except Exception as e:
        print(f"Error processing videos {video_ids_str}: {e}")
        return None

# Create the engine
engine = create_engine('your_database_connection_string')
inspector = inspect(engine)

# Main logic
if (has_channel_playlists := inspector.has_table('channel_playlists')):

    iteration = 0

    while True:
        channel_playlists_df = pd.read_sql_table('channel_playlists', engine)

        if inspector.has_table('video_statistics'):
            video_statistics_df = pd.read_sql_table('video_statistics', engine)
            video_statistics_video_ids = video_statistics_df['id'].unique()
            channel_playlists_df = channel_playlists_df[~channel_playlists_df['contentDetails.videoId'].isin(video_statistics_video_ids)]

        len_videos = channel_playlists_df.shape[0]

        if len_videos == 0:
            print("All videos fetched.")
            break
        else:
            print(f"Iteration {iteration} with {len_videos} videos remaining")

        video_ids = channel_playlists_df['snippet.resourceId.videoId'].unique()

        # Break into batches of 50
        batch_size = 50
        video_id_batches = [','.join(video_ids[i:i + batch_size]) for i in range(0, len(video_ids), batch_size)]

        # Prepare arguments for process_batch
        args_list = [(video_ids_str, API_KEYS) for video_ids_str in video_id_batches]

        num_processes = 4  # Adjust based on your CPU and API quotas

        # Use ProcessPoolExecutor with as_completed for better progress tracking
        with ProcessPoolExecutor(max_workers=num_processes) as executor:
            futures = {executor.submit(process_batch, arg): arg for arg in args_list}

            # Use tqdm to display the progress bar
            data_frames = []
            for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching Video Statistics"):
                result = future.result()
                if result is not None:
                    data_frames.append(result)

        if data_frames:
            combined_df = pd.concat(data_frames, ignore_index=True)

            # Add missing columns before inserting the data
            add_missing_columns(engine, 'video_statistics', combined_df)

            # Save the data to the database
            combined_df.to_sql('video_statistics', engine, if_exists='append', index=False)
        else:
            print("No data frames to save.")

        iteration += 1


In [ ]:
# [*] Parallel video statistics fetch
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
from sqlalchemy import inspect
import random
import json

from googleapiclient.errors import HttpError
from googleapiclient.discovery import build


# Get video statistics
tqdm.pandas()

def get_channel_videos_df(playlist_id, api_keys, current_key_index, pbar, verbose=False):
    
    next_page_token = None 
    current_video_count = 0
    video_stats = []
    
    while True:
        
        message = f'Current Video ID Count: {current_video_count}, Current API Key Index: {current_key_index}'
        pbar.set_postfix_str(message)

        try:
            youtube = build('youtube', 'v3', developerKey=api_keys[current_key_index])

            playlist_items = youtube.playlistItems().list(
                part='contentDetails,snippet',
                playlistId=playlist_id,
                maxResults=50,
                pageToken=next_page_token
            ).execute()

            next_page_token = playlist_items.get('nextPageToken')

            video_ids = [item['snippet']['resourceId']['videoId'] for item in playlist_items['items']]
            video_id_string = ','.join(video_ids)

            response = youtube.videos().list(
                part='contentDetails,id,liveStreamingDetails,snippet,statistics,topicDetails',
                id=video_id_string
            ).execute()
            
            items = response.get('items')
            current_video_count += len(items)
            video_stats.extend(items)
            
            pbar.update(len(items))

            if not next_page_token:
                break
            
        except HttpError as e:
            if e.resp.status == 403:  # Quota exceeded
                current_key_index = (current_key_index + 1) % len(api_keys)
                if current_key_index == 0:
                    print("All API keys exhausted.")
                    break
            elif e.resp.status == 404:
                print("Playlist not found.")
                break
            else:
                raise  # Re-raise the exception if it's not a quota error
            
    video_stats_df = pd.json_normalize(video_stats)
    
    # video_stats_df = process_stats_data(video_stats_df)
    
    return video_stats_df, current_key_index

def get_channel_videos_df_parallel(channel_id, api_keys, current_key_index, pbar, verbose=False):
    try:
        playlist_id = 'UU' + channel_id[2:]
        return get_channel_videos_df(playlist_id, api_keys, current_key_index, pbar, verbose)
    except TimeoutError as e:
        print(f"Timeout error on channel {channel_id}: {e}")
    except HttpError as e:
        print(f"Http error on channel {channel_id}: {e}")
    return None, current_key_index

from sqlalchemy import text

def add_new_columns(engine, table_name, new_columns):
    with engine.connect() as connection:
        for column in new_columns:
            try:
                # Use SQLAlchemy's `text()` for raw SQL execution
                connection.execute(text(f'ALTER TABLE {table_name} ADD COLUMN "{column}" TEXT'))
            except Exception as e:
                # Catch any exception (like if the column already exists)
                print(f"Error adding column {column}: {e}")
                
def get_existing_columns(engine, table_name):
    with engine.connect() as connection:
        # Use SQLAlchemy's `text()` for the PRAGMA query
        result = connection.execute(text(f"PRAGMA table_info({table_name})")).fetchall()
        # Access the second element (index 1) of each tuple for the column name
        existing_column_names = [row[1] for row in result]
        return existing_column_names

def parallel_channel_video_fetch(api_keys, current_key_index, channel_statistics_df, engine):
    
    videos_df = (
        pd.read_sql_table('video_statistics', engine).set_index('id')
        if inspector.has_table('video_statistics')
        else pd.DataFrame(columns=['id', 'snippet.channelId']).set_index('id')
    )
    
    # Get unprocessed channel IDs
    channel_statistics_channel_ids = channel_statistics_df.index
    
    print(f"Total channel statistics table IDs: {len(channel_statistics_channel_ids)}")
   
    video_statistics_channel_ids = videos_df['snippet.channelId'].unique()
    
    print(f"Total video statistics table IDs: {len(video_statistics_channel_ids)}")
    
    channel_ids = list(set(channel_statistics_channel_ids) - set(video_statistics_channel_ids))
    
    print(f"Total channel IDs to fetch: {len(channel_ids)}")
    
    if len(channel_ids) == 0:
        
        print("No new channel IDs to fetch.")
        
        return pd.read_sql_table('video_statistics', engine)
    
    random.shuffle(channel_ids)  # Shuffle to avoid hitting the same API keys quickly
   
    # Use a ThreadPoolExecutor for parallel requests
    with ThreadPoolExecutor(max_workers=2) as executor:  # Tune max_workers to API and network limits
        futures = []
        
        pbar_len = channel_statistics_df.loc[channel_ids, 'statistics.videoCount'].astype(int).sum()
        
        print(f"Total videos to fetch: {pbar_len}")
        
        pbar = tqdm(total=int(pbar_len), desc="Fetching Videos")
        
        # Submit tasks to executor
        for channel_id in channel_ids:
            futures.append(executor.submit(get_channel_videos_df_parallel, channel_id, api_keys, current_key_index, pbar))

        # Collect results as they complete
        for future in as_completed(futures):
            channel_videos_df, current_key_index = future.result()
            if channel_videos_df is not None:
                try:
                    channel_id = channel_videos_df['id'].iloc[0]

                    if len(channel_videos_df) == 0:
                        channel_statistics_df.drop(channel_id, inplace=True)
                        channel_statistics_df.to_sql('channel_statistics', engine, if_exists='replace', index=True)
                        print(f"Channel {channel_id} has no relevant videos. Dropped from channel_statistics table.")
                    else:
                        # Convert list and dict columns to JSON strings
                        for column in channel_videos_df.columns:
                            if channel_videos_df[column].apply(lambda x: isinstance(x, (list, dict))).any():
                                channel_videos_df[column] = channel_videos_df[column].apply(json.dumps)

                        # Retrieve existing columns in the table
                        existing_columns = get_existing_columns(engine, 'video_statistics')
                        new_columns = set(channel_videos_df.columns) - set(existing_columns)

                        if new_columns and inspector.has_table('video_statistics'):
                            # Add any new columns dynamically
                            add_new_columns(engine, 'video_statistics', new_columns)

                        # Now, append the data to the database
                        channel_videos_df.to_sql('video_statistics', engine, if_exists='append', index=True)

                except IndexError:
                    print(f'IndexError: {channel_id}')

                    
    vid_stats = pd.read_sql_table('video_statistics', engine)
    
    vid_stats_ids = vid_stats['id'].unique()
    
    print(f"Downloaded data for number of IDs: {len(vid_stats_ids)}")

    print(vid_stats)
                    
    print(vid_stats.shape)

    print()
    
    channel_statistics_df = pd.read_sql_table('channel_statistics', engine, index_col='id')
    
    parallel_channel_video_fetch(api_keys, current_key_index, channel_statistics_df, engine)

# Create the engine
engine = create_engine(f'sqlite:///../data/data.db', echo=True)
inspector = inspect(engine)
                    
# Main logic
if (has_channel_statistics := inspector.has_table('channel_statistics')):
    channel_statistics_df = pd.read_sql_table('channel_statistics', engine, index_col='id')
    
    # Call the parallelized video fetching function
    current_key_index = 0
    
    vid_stats_df = parallel_channel_video_fetch(API_KEYS, current_key_index, channel_statistics_df, engine)
    
    print(vid_stats_df)

In [ ]:
# read video_statistics table
# video_statistics_df = pd.read_sql_table('video_statistics', engine)
video_statistics_df = pd.read_sql_table('channel_playlists', engine)

video_statistics_df

In [ ]:
from sqlalchemy import create_engine, inspect, MetaData, Table, Column

# Create the engine
db_loc = 'sqlite:///../data/db/fn_m3_split.db'
engine = create_engine(db_loc, echo=True)
inspector = inspect(engine)

print(inspector.get_table_names())

# import pandas as pd

# df = pd.read_sql_table('video_statistics', engine)
# df

In [ ]:
%%time


table_size = pd.read_sql_query("SELECT COUNT(*) FROM channel_playlists", engine)

table_size

In [ ]:
table_size.values[0][0]